# Lumen take-home: July cost gap, unit economics, onboarding_v2
All timestamps are UTC. The invoice is my cost basis (section 2 shows why).

Sections: 0 Setup, 1 Data checks, 2 Reconcile July, 3 Unit economics, 4 Experiment, 5 Limits, Appendix

## 0. Setup

In [1]:
import os, duckdb
p = "lumen.duckdb"
print(os.path.getsize(p) / 1e6, "MB") 
con = duckdb.connect(p, read_only=True)
con.sql("SET TimeZone='UTC'")
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) FROM usage_events"))

12.333056 MB
┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ experiment_assignments  │
│ finance_dashboard_daily │
│ model_pricing           │
│ provider_invoices       │
│ subscription_events     │
│ usage_events            │
│ users                   │
└─────────────────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       308539 │
└──────────────┘



## 1. Data checks
### 1a. The small tables

In [3]:
con.sql("SELECT * FROM model_pricing").show(max_rows=50, max_width=200)
con.sql("SELECT * FROM provider_invoices").show(max_rows=50, max_width=200)
con.sql("DESCRIBE finance_dashboard_daily").show(max_width=200)
con.sql("SELECT * FROM finance_dashboard_daily LIMIT 10").show(max_width=200)

┌──────────┬──────────────┬──────────────────────┬───────────┬────────────────┬─────────────┐
│ provider │    model     │         unit         │ price_usd │ effective_from │ updated_by  │
│ varchar  │   varchar    │       varchar        │  double   │      date      │   varchar   │
├──────────┼──────────────┼──────────────────────┼───────────┼────────────────┼─────────────┤
│ vox      │ vox-stt-2    │ per_audio_minute     │     0.006 │ 2026-01-01     │ finance-ops │
│ nimbus   │ nimbus-large │ per_1m_input_tokens  │       3.0 │ 2026-01-01     │ finance-ops │
│ nimbus   │ nimbus-large │ per_1m_output_tokens │      15.0 │ 2026-01-01     │ finance-ops │
│ nimbus   │ nimbus-mini  │ per_1m_input_tokens  │      0.15 │ 2026-01-01     │ finance-ops │
│ nimbus   │ nimbus-mini  │ per_1m_output_tokens │       0.6 │ 2026-01-01     │ finance-ops │
│ nimbus   │ nimbus-embed │ per_1m_input_tokens  │      0.02 │ 2026-01-01     │ finance-ops │
└──────────┴──────────────┴──────────────────────┴──────────

### 1b. Starting totals, July only

In [5]:
con.sql("""SELECT ROUND(SUM(amount_usd),2) AS july_invoice FROM provider_invoices
           WHERE billing_period_start = DATE '2026-07-01'""").show()
con.sql("""SELECT ROUND(SUM(cost_usd),2) AS july_dashboard FROM finance_dashboard_daily
           WHERE date_pt BETWEEN DATE '2026-07-01' AND DATE '2026-07-31'""").show()

┌──────────────┐
│ july_invoice │
│    double    │
├──────────────┤
│      6860.56 │
└──────────────┘

┌────────────────┐
│ july_dashboard │
│     double     │
├────────────────┤
│        6310.49 │
└────────────────┘



### 1c. Duplicates, status mix, models, ingestion lag

In [7]:
con.sql("""
SELECT COUNT(*) - COUNT(DISTINCT event_id) AS dup_event_ids,
       COUNT(*) - COUNT(DISTINCT request_id) AS dup_request_ids
FROM usage_events
""").show()

con.sql("SELECT status, COUNT(*) n FROM usage_events GROUP BY 1").show()
con.sql("SELECT provider, model, feature, COUNT(*) n FROM usage_events GROUP BY 1,2,3 ORDER BY 1,2,3").show(max_rows=60, max_width=200)
con.sql("SELECT MEDIAN(ingested_ts - event_ts) med_lag, MAX(ingested_ts - event_ts) max_lag FROM usage_events").show()

┌───────────────┬─────────────────┐
│ dup_event_ids │ dup_request_ids │
│     int64     │      int64      │
├───────────────┼─────────────────┤
│             0 │            2509 │
└───────────────┴─────────────────┘

┌─────────┬────────┐
│ status  │   n    │
│ varchar │ int64  │
├─────────┼────────┤
│ success │ 301096 │
│ timeout │   4078 │
│ error   │   3365 │
└─────────┴────────┘

┌──────────┬───────────────────┬─────────────────┬───────┐
│ provider │       model       │     feature     │   n   │
│ varchar  │      varchar      │     varchar     │ int64 │
├──────────┼───────────────────┼─────────────────┼───────┤
│ nimbus   │ nimbus-embed      │ embed           │ 85621 │
│ nimbus   │ nimbus-large      │ ask             │  6922 │
│ nimbus   │ nimbus-large      │ onboarding_demo │  1476 │
│ nimbus   │ nimbus-large      │ summarize       │ 77824 │
│ nimbus   │ nimbus-large-0526 │ ask             │   827 │
│ nimbus   │ nimbus-large-0526 │ onboarding_demo │    53 │
│ nimbus   │ nimbus-larg

308,539 events. No duplicate event_id, but 2,509 repeated request_ids, and the repeats are identical copies. Max ingestion lag is about 3.5 minutes, so late data can't explain the gap. nimbus-large-0526 shows up in the logs and not in model_pricing.

## 2. Reconcile July
### 2a. What the dashboard says

In [10]:
# 1. Dashboard July total and by category
con.sql("""
SELECT category, ROUND(SUM(cost_usd),2) AS usd
FROM finance_dashboard_daily
WHERE date_pt BETWEEN '2026-07-01' AND '2026-07-31'
GROUP BY ROLLUP(category) ORDER BY 1
""").show()

# 2. June control: dashboard June vs invoice gross June
con.sql("""
SELECT ROUND(SUM(cost_usd),2) AS dashboard_june
FROM finance_dashboard_daily
WHERE date_pt BETWEEN '2026-06-01' AND '2026-06-30'
""").show()

# 3. July logged units by model (UTC month), all statuses vs success only
con.sql("""
SELECT model, status,
       COUNT(*) AS n,
       SUM(input_tokens) AS in_tok,
       SUM(output_tokens) AS out_tok,
       ROUND(SUM(audio_seconds)/60.0, 2) AS audio_min_exact
FROM usage_events
WHERE event_ts >= '2026-07-01' AND event_ts < '2026-08-01'
GROUP BY 1,2 ORDER BY 1,2
""").show(max_rows=60, max_width=200)

# 4. Vox rounding: exact minutes vs per-request 15-second increments
con.sql("""
SELECT status,
       ROUND(SUM(audio_seconds)/60.0, 2) AS exact_min,
       ROUND(SUM(CEIL(audio_seconds/15.0)*15)/60.0, 2) AS rounded_min
FROM usage_events
WHERE provider='vox'
  AND event_ts >= '2026-07-01' AND event_ts < '2026-08-01'
GROUP BY 1
""").show()

# 5. What do duplicate request_ids look like?
con.sql("""
WITH d AS (SELECT request_id FROM usage_events GROUP BY 1 HAVING COUNT(*) > 1)
SELECT u.request_id, u.event_id, u.event_ts, u.ingested_ts, u.feature, u.model, u.status,
       u.audio_seconds, u.input_tokens, u.output_tokens
FROM usage_events u JOIN d USING (request_id)
ORDER BY u.request_id, u.event_ts
LIMIT 20
""").show(max_width=250)

┌───────────────┬─────────┐
│   category    │   usd   │
│    varchar    │ double  │
├───────────────┼─────────┤
│ chat          │  161.25 │
│ embeddings    │    2.93 │
│ summarization │ 1711.21 │
│ transcription │  4435.1 │
│ NULL          │ 6310.49 │
└───────────────┴─────────┘

┌────────────────┐
│ dashboard_june │
│     double     │
├────────────────┤
│        6502.43 │
└────────────────┘

┌───────────────────┬─────────┬───────┬───────────┬──────────┬─────────────────┐
│       model       │ status  │   n   │  in_tok   │ out_tok  │ audio_min_exact │
│      varchar      │ varchar │ int64 │  int128   │  int128  │     double      │
├───────────────────┼─────────┼───────┼───────────┼──────────┼─────────────────┤
│ nimbus-embed      │ error   │   228 │    768846 │     NULL │            NULL │
│ nimbus-embed      │ success │ 43302 │ 147800618 │        0 │            NULL │
│ nimbus-large      │ error   │   635 │   6751448 │     NULL │            NULL │
│ nimbus-large      │ success │ 38351

### 2b. What the providers actually bill
Deduplicated view first, then billed units next to the invoice line items.

In [12]:
con.sql("SET TimeZone='UTC'")

# Deduplicated events: keep the first-ingested copy of each request_id
con.sql("""
CREATE OR REPLACE TEMP VIEW ev AS
SELECT * FROM usage_events
QUALIFY ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ingested_ts) = 1
""")

In [14]:
# 1. Billed units under the hypothesis, July UTC.
con.sql("""
WITH e AS (
  SELECT *, CASE WHEN model LIKE 'nimbus-large%' THEN 'nimbus-large' ELSE model END AS billed_model
  FROM ev
  WHERE event_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00'
    AND event_ts <  TIMESTAMPTZ '2026-08-01 00:00:00+00'
)
SELECT billed_model,
  SUM(CASE WHEN status IN ('success','timeout') THEN input_tokens END) AS in_succ_plus_timeout,
  SUM(CASE WHEN status = 'success' THEN input_tokens END)              AS in_success_only,
  SUM(CASE WHEN status = 'success' THEN output_tokens END)             AS out_success,
  ROUND(SUM(CASE WHEN status IN ('success','timeout')
                 THEN CEIL(audio_seconds/15.0)*15 END)/60.0, 2)        AS vox_min_succ_timeout
FROM e GROUP BY 1 ORDER BY 1
""").show(max_width=200)

# 2. Are duplicate rows identical copies? (expect 0 rows differing)
con.sql("""
SELECT COUNT(*) AS request_ids_with_differences FROM (
  SELECT request_id FROM usage_events GROUP BY 1
  HAVING COUNT(*) > 1 AND (COUNT(DISTINCT status) > 1
     OR COUNT(DISTINCT COALESCE(output_tokens,-1)) > 1
     OR COUNT(DISTINCT COALESCE(audio_seconds,-1)) > 1)
)
""").show()

# 3. When does nimbus-large-0526 first appear, and which app versions?
con.sql("""
SELECT MIN(event_ts) AS first_seen, MAX(event_ts) AS last_seen,
       COUNT(*) AS n, COUNT(DISTINCT app_version) AS versions
FROM usage_events WHERE model = 'nimbus-large-0526'
""").show(max_width=200)

# 4. Does the Jul 1-14 / Jul 15-31 output split match the invoice? Targets: 30,174,436 and 20,097,053
con.sql("""
SELECT event_ts >= TIMESTAMPTZ '2026-07-15 00:00:00+00' AS from_jul15,
       SUM(output_tokens) AS out_tokens
FROM ev
WHERE model LIKE 'nimbus-large%' AND status = 'success'
  AND event_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00'
  AND event_ts <  TIMESTAMPTZ '2026-08-01 00:00:00+00'
GROUP BY 1 ORDER BY 1
""").show()

┌──────────────┬──────────────────────┬─────────────────┬─────────────┬──────────────────────┐
│ billed_model │ in_succ_plus_timeout │ in_success_only │ out_success │ vox_min_succ_timeout │
│   varchar    │        int128        │     int128      │   int128    │        double        │
├──────────────┼──────────────────────┼─────────────────┼─────────────┼──────────────────────┤
│ nimbus-embed │            145603339 │       145603339 │           0 │                 NULL │
│ nimbus-large │            502839281 │       492656483 │    50271489 │                 NULL │
│ nimbus-mini  │            229096160 │       224077944 │     7633335 │                 NULL │
│ vox-stt-2    │                 NULL │            NULL │        NULL │             749303.0 │
└──────────────┴──────────────────────┴─────────────────┴─────────────┴──────────────────────┘

┌──────────────────────────────┐
│ request_ids_with_differences │
│            int64             │
├──────────────────────────────┤
│           

Logged units match the invoice on all six lines. So I know the billing rules: repeated requests are billed once, errors never, timeouts yes, Vox rounds up to 15 seconds per request, and nimbus-large-0526 is billed as nimbus-large.

### 2c. How the dashboard works
Same calculation under three status filters, compared by category.

In [17]:
con.sql("SET TimeZone='UTC'")
con.sql("""
WITH b AS (
  SELECT *, CAST(timezone('America/Los_Angeles', event_ts) AS DATE) AS pt_date,
    CASE WHEN model LIKE 'nimbus-large%' THEN 'nimbus-large' ELSE model END AS bm
  FROM usage_events
),
c AS (
  SELECT *,
    COALESCE(CASE WHEN provider='vox' THEN audio_seconds/60.0*0.006 END,0)
    + COALESCE(CASE bm WHEN 'nimbus-large' THEN 3.0 WHEN 'nimbus-mini' THEN 0.15 WHEN 'nimbus-embed' THEN 0.02 END/1e6*input_tokens,0)
    + COALESCE(CASE bm WHEN 'nimbus-large' THEN 15.0 WHEN 'nimbus-mini' THEN 0.6 ELSE 0 END/1e6*output_tokens,0) AS cost
  FROM b WHERE pt_date BETWEEN DATE '2026-07-01' AND DATE '2026-07-31'
)
SELECT feature,
  ROUND(SUM(cost) FILTER (WHERE model<>'nimbus-large-0526'),2)                                         AS all_status,
  ROUND(SUM(cost) FILTER (WHERE model<>'nimbus-large-0526' AND status='success'),2)                    AS success_only,
  ROUND(SUM(cost) FILTER (WHERE model<>'nimbus-large-0526' AND status IN ('success','timeout')),2)      AS succ_timeout
FROM c GROUP BY ROLLUP(feature) ORDER BY 1
""").show(max_width=200)

┌─────────────────┬────────────┬──────────────┬──────────────┐
│     feature     │ all_status │ success_only │ succ_timeout │
│     varchar     │   double   │    double    │    double    │
├─────────────────┼────────────┼──────────────┼──────────────┤
│ ask             │     166.45 │       161.28 │       164.16 │
│ embed           │       2.95 │         2.93 │         2.93 │
│ onboarding_demo │      15.37 │         15.1 │        15.27 │
│ summarize       │    1737.26 │      1696.13 │      1719.08 │
│ transcribe      │    4539.78 │       4435.1 │      4503.54 │
│ NULL            │    6461.81 │      6310.54 │      6404.98 │
└─────────────────┴────────────┴──────────────┴──────────────┘



### 2d. Bridge from dashboard to invoice
Each column changes one thing from the one before. Ordered as listed.

In [19]:
con.sql("SET TimeZone='UTC'")
con.sql("""
WITH b AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ingested_ts) AS rn,
    CAST(timezone('America/Los_Angeles', event_ts) AS DATE) AS pt_date,
    CASE WHEN model LIKE 'nimbus-large%' THEN 'nimbus-large' ELSE model END AS bm
  FROM usage_events
),
c AS (
  SELECT *,
    COALESCE(CASE WHEN provider='vox' THEN audio_seconds/60.0*0.006 END,0) AS vox_exact,
    COALESCE(CASE WHEN provider='vox' THEN CEIL(audio_seconds/15.0)*15/60.0*0.006 END,0) AS vox_round,
    COALESCE(CASE bm WHEN 'nimbus-large' THEN 3.0 WHEN 'nimbus-mini' THEN 0.15
                     WHEN 'nimbus-embed' THEN 0.02 END/1e6*input_tokens,0) AS in_cost,
    COALESCE(CASE bm WHEN 'nimbus-large' THEN 15.0 WHEN 'nimbus-mini' THEN 0.6 ELSE 0 END/1e6*output_tokens,0) AS out_cost,
    CASE WHEN bm='nimbus-large' AND event_ts >= TIMESTAMPTZ '2026-07-15 00:00:00+00'
         THEN 3.0/1e6*COALESCE(output_tokens,0) ELSE 0 END AS uplift,
    (pt_date BETWEEN DATE '2026-07-01' AND DATE '2026-07-31') AS pt_july,
    (event_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00'
     AND event_ts <  TIMESTAMPTZ '2026-08-01 00:00:00+00') AS utc_july
  FROM b
)
SELECT
 ROUND(SUM(vox_exact+in_cost+out_cost) FILTER (WHERE pt_july AND status='success' AND model<>'nimbus-large-0526'),2) AS A_dashboard_rules,
 ROUND(SUM(vox_exact+in_cost+out_cost) FILTER (WHERE pt_july AND status='success'),2) AS B_add_0526,
 ROUND(SUM(vox_exact+in_cost+out_cost+uplift) FILTER (WHERE pt_july AND status='success'),2) AS C_price_rise,
 ROUND(SUM(vox_exact+in_cost+out_cost+uplift) FILTER (WHERE pt_july AND status='success' AND rn=1),2) AS D_dedupe,
 ROUND(SUM(vox_exact+in_cost+out_cost+uplift) FILTER (WHERE pt_july AND status IN ('success','timeout') AND rn=1),2) AS E_add_timeouts,
 ROUND(SUM(vox_round+in_cost+out_cost+uplift) FILTER (WHERE pt_july AND status IN ('success','timeout') AND rn=1),2) AS F_vox_rounding,
 ROUND(SUM(vox_round+in_cost+out_cost+uplift) FILTER (WHERE utc_july AND status IN ('success','timeout') AND rn=1),2) AS G_utc_month
FROM c
""").show(max_width=250)

┌───────────────────┬────────────┬──────────────┬──────────┬────────────────┬────────────────┬─────────────┐
│ A_dashboard_rules │ B_add_0526 │ C_price_rise │ D_dedupe │ E_add_timeouts │ F_vox_rounding │ G_utc_month │
│      double       │   double   │    double    │  double  │     double     │     double     │   double    │
├───────────────────┼────────────┼──────────────┼──────────┼────────────────┼────────────────┼─────────────┤
│           6310.54 │    6728.34 │      6790.39 │  6679.58 │        6777.47 │        6808.88 │     6860.56 │
└───────────────────┴────────────┴──────────────┴──────────┴────────────────┴────────────────┴─────────────┘



The (\\$ 550.07) gap is fully covered. The two big pieces are the unpriced model (+\\$417.80) and repeated requests (-$110.81), which pull in opposite directions. Step sizes depend on the order, so read them as ordered as listed. The invoice is the right number.

## 3. Unit economics, July
### 3a. Tables for revenue and users

In [21]:
con.sql("DESCRIBE users").show()
con.sql("DESCRIBE subscription_events").show()
con.sql("SELECT * FROM subscription_events LIMIT 8").show(max_width=200)

┌────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │       column_type        │  null   │   key   │ default │  extra  │
│      varchar       │         varchar          │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ user_id            │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ anon_id            │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ account_created_ts │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ platform           │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ app_version        │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ country            │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────┴──────────────────────────┴─────────┴─────────┴─────────┴─────────┘

┌────────

### 3b. Cost basis and category totals
One row per request_id, success and timeout only, Vox rounded up to 15 seconds, the Jul 15 output price, UTC months. The check must print 6860.56.

In [23]:
con.sql("SET TimeZone='UTC'")
con.sql("""
CREATE OR REPLACE TEMP VIEW cost AS
SELECT request_id, user_id, feature, status, event_ts, input_tokens,
       CAST(event_ts AS DATE) AS d,
       COALESCE(CASE WHEN provider='vox' THEN CEIL(audio_seconds/15.0)*15/60.0*0.006 END,0)
     + COALESCE(CASE WHEN model LIKE 'nimbus-large%' THEN 3.0 WHEN model='nimbus-mini' THEN 0.15
                     WHEN model='nimbus-embed' THEN 0.02 END/1e6*input_tokens,0)
     + COALESCE(CASE WHEN model LIKE 'nimbus-large%' THEN 15.0 WHEN model='nimbus-mini' THEN 0.6 ELSE 0 END/1e6*output_tokens,0)
     + CASE WHEN model LIKE 'nimbus-large%' AND event_ts >= TIMESTAMPTZ '2026-07-15 00:00:00+00'
            THEN 3.0/1e6*COALESCE(output_tokens,0) ELSE 0 END AS usd
FROM usage_events
QUALIFY ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ingested_ts) = 1
   AND status IN ('success','timeout')
""")

# sanity check, must still be 6860.56
con.sql("""SELECT ROUND(SUM(usd),2) FROM cost
WHERE event_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00' AND event_ts < TIMESTAMPTZ '2026-08-01 00:00:00+00'""").show()

┌────────────────────┐
│ round(sum(usd), 2) │
│       double       │
├────────────────────┤
│            6860.56 │
└────────────────────┘



In [25]:
con.sql("""
SELECT feature, ROUND(SUM(usd),2) AS usd, ROUND(100*SUM(usd)/SUM(SUM(usd)) OVER (),1) AS pct
FROM cost WHERE d BETWEEN DATE '2026-07-01' AND DATE '2026-07-31'
GROUP BY 1 ORDER BY 2 DESC
""").show()

┌─────────────────┬─────────┬────────┐
│     feature     │   usd   │  pct   │
│     varchar     │ double  │ double │
├─────────────────┼─────────┼────────┤
│ transcribe      │ 4495.82 │   65.5 │
│ summarize       │ 2150.45 │   31.3 │
│ ask             │  194.49 │    2.8 │
│ onboarding_demo │   16.89 │    0.2 │
│ embed           │    2.91 │    0.0 │
└─────────────────┴─────────┴────────┘



July AI cost is $6,860.56. Transcription is 65.5%, summarization 31.3%, ask 2.8%. Demo and embeddings are tiny.

### 3c. Active users and cost per user
Active user: at least one successful, non-demo event on that UTC day. There are no app-open events, so this only counts people who called a provider.

In [27]:
con.sql("""
WITH dau AS (
  SELECT d, COUNT(DISTINCT user_id) AS dau FROM cost
  WHERE status='success' AND feature<>'onboarding_demo'
    AND d BETWEEN DATE '2026-07-01' AND DATE '2026-07-31'
  GROUP BY 1),
dc AS (
  SELECT d, SUM(usd) AS usd FROM cost
  WHERE d BETWEEN DATE '2026-07-01' AND DATE '2026-07-31' GROUP BY 1)
SELECT ROUND(AVG(dau),0) AS avg_dau,
       ROUND(SUM(usd)/SUM(dau),4) AS cost_per_dau_day,
       ROUND(AVG(usd/dau),4) AS avg_of_daily_ratios
FROM dau JOIN dc USING (d)
""").show()

# Monthly view and skew
con.sql("""
WITH u AS (
  SELECT user_id, SUM(usd) AS usd FROM cost
  WHERE d BETWEEN DATE '2026-07-01' AND DATE '2026-07-31' AND user_id IS NOT NULL
  GROUP BY 1)
SELECT COUNT(*) AS users,
       ROUND(AVG(usd),3) AS mean_usd, ROUND(MEDIAN(usd),3) AS median_usd,
       ROUND(QUANTILE_CONT(usd,0.99),2) AS p99_usd,
       ROUND(100*SUM(usd) FILTER (WHERE usd >= (SELECT QUANTILE_CONT(usd,0.99) FROM u))/SUM(usd),1) AS top1pct_share
FROM u
""").show()

┌─────────┬──────────────────┬─────────────────────┐
│ avg_dau │ cost_per_dau_day │ avg_of_daily_ratios │
│ double  │      double      │       double        │
├─────────┼──────────────────┼─────────────────────┤
│   985.0 │           0.2246 │              0.2249 │
└─────────┴──────────────────┴─────────────────────┘

┌───────┬──────────┬────────────┬─────────┬───────────────┐
│ users │ mean_usd │ median_usd │ p99_usd │ top1pct_share │
│ int64 │  double  │   double   │ double  │    double     │
├───────┼──────────┼────────────┼─────────┼───────────────┤
│  8403 │    0.816 │      0.477 │    4.81 │           7.4 │
└───────┴──────────┴────────────┴─────────┴───────────────┘



About 985 active users a day, at $0.22 each per day. The top 1% of users make up only 7.4% of cost, so there's no heavy-user problem to fix.

### 3d. First-week cost per new user
June signups vs July signups through Jul 24, so everyone has 7 full days. Divided by all new users.

In [29]:
con.sql("""
WITH n AS (
  SELECT user_id, account_created_ts AS t0,
    CASE WHEN account_created_ts >= TIMESTAMPTZ '2026-06-01 00:00:00+00' AND account_created_ts < TIMESTAMPTZ '2026-07-01 00:00:00+00' THEN 'June'
         WHEN account_created_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00' AND account_created_ts < TIMESTAMPTZ '2026-07-25 00:00:00+00' THEN 'July' END AS cohort
  FROM users),
w AS (
  SELECT n.cohort, n.user_id, COALESCE(SUM(c.usd),0) AS wk1
  FROM n LEFT JOIN cost c ON c.user_id=n.user_id AND c.event_ts >= n.t0 AND c.event_ts < n.t0 + INTERVAL 7 DAY
  WHERE n.cohort IS NOT NULL GROUP BY 1,2)
SELECT cohort, COUNT(*) AS new_users, ROUND(AVG(wk1),3) AS mean_wk1_cost,
       ROUND(MEDIAN(wk1),3) AS median_wk1_cost, ROUND(QUANTILE_CONT(wk1,0.95),3) AS p95
FROM w GROUP BY 1 ORDER BY 1
""").show()

┌─────────┬───────────┬───────────────┬─────────────────┬────────┐
│ cohort  │ new_users │ mean_wk1_cost │ median_wk1_cost │  p95   │
│ varchar │   int64   │    double     │     double      │ double │
├─────────┼───────────┼───────────────┼─────────────────┼────────┤
│ July    │      2506 │         0.603 │           0.411 │  1.846 │
│ June    │      4494 │         0.613 │           0.423 │  1.912 │
└─────────┴───────────┴───────────────┴─────────────────┴────────┘



\\$ 0.613 for June, $0.603 for July. Basically flat. July signups also ran about 30% below June. I didn't look into why.

### 3e. Revenue vs AI cost
Revenue is proceeds_usd, net of refunds.

In [31]:
con.sql("""
SELECT event_type, product_id, store, COUNT(*) n,
       ROUND(SUM(price_usd),2) price, ROUND(SUM(proceeds_usd),2) proceeds
FROM subscription_events
WHERE event_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00' AND event_ts < TIMESTAMPTZ '2026-08-01 00:00:00+00'
GROUP BY 1,2,3 ORDER BY 1,2,3
""").show(max_rows=60, max_width=200)

┌─────────────────┬───────────────────┬────────────┬───────┬─────────┬──────────┐
│   event_type    │    product_id     │   store    │   n   │  price  │ proceeds │
│     varchar     │      varchar      │  varchar   │ int64 │ double  │  double  │
├─────────────────┼───────────────────┼────────────┼───────┼─────────┼──────────┤
│ cancellation    │ lumen_pro_annual  │ app_store  │    60 │     0.0 │      0.0 │
│ cancellation    │ lumen_pro_annual  │ play_store │    35 │     0.0 │      0.0 │
│ cancellation    │ lumen_pro_annual  │ stripe     │    12 │     0.0 │      0.0 │
│ cancellation    │ lumen_pro_monthly │ app_store  │   196 │     0.0 │      0.0 │
│ cancellation    │ lumen_pro_monthly │ play_store │    96 │     0.0 │      0.0 │
│ cancellation    │ lumen_pro_monthly │ stripe     │    29 │     0.0 │      0.0 │
│ refund          │ lumen_pro_annual  │ app_store  │     1 │ -119.99 │  -101.99 │
│ refund          │ lumen_pro_annual  │ play_store │     2 │ -239.98 │  -203.98 │
│ renewal       

In [33]:
con.sql("""
SELECT ROUND(SUM(proceeds_usd),2) AS net_proceeds,
       ROUND(SUM(proceeds_usd) FILTER (WHERE product_id='lumen_pro_monthly'),2) AS monthly_only
FROM subscription_events
WHERE event_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00'
  AND event_ts <  TIMESTAMPTZ '2026-08-01 00:00:00+00'
""").show()

┌──────────────┬──────────────┐
│ net_proceeds │ monthly_only │
│    double    │    double    │
├──────────────┼──────────────┤
│     17377.97 │      9600.74 │
└──────────────┴──────────────┘



Net cash proceeds were $17,377.97, so AI cost is 39.5% of cash. That flatters July, because annual plans are paid up front. On monthly plans alone it's about 71%. The real figure sits somewhere between the two.

### 3f. Who drives cost, and where to save
Cost by user segment first, then the size of summarize inputs.

In [35]:
con.sql("""
WITH seg AS (
  SELECT user_id,
    MAX(CASE WHEN event_type IN ('trial_converted','renewal') THEN 1 ELSE 0 END) AS paid,
    MAX(CASE WHEN event_type = 'trial_started' THEN 1 ELSE 0 END) AS trialed
  FROM subscription_events WHERE event_ts < TIMESTAMPTZ '2026-08-01 00:00:00+00'
  GROUP BY 1),
u AS (
  SELECT user_id, SUM(usd) AS usd FROM cost
  WHERE d BETWEEN DATE '2026-07-01' AND DATE '2026-07-31' AND user_id IS NOT NULL GROUP BY 1)
SELECT CASE WHEN s.paid=1 THEN 'paid' WHEN s.trialed=1 THEN 'trial only' ELSE 'no subscription' END AS segment,
       COUNT(*) users, ROUND(SUM(u.usd),2) AS usd, ROUND(AVG(u.usd),3) AS avg_usd
FROM u LEFT JOIN seg s USING (user_id)
GROUP BY 1 ORDER BY 3 DESC
""").show()

┌─────────────────┬───────┬─────────┬─────────┐
│     segment     │ users │   usd   │ avg_usd │
│     varchar     │ int64 │ double  │ double  │
├─────────────────┼───────┼─────────┼─────────┤
│ no subscription │  7414 │ 5709.56 │    0.77 │
│ paid            │   575 │  680.67 │   1.184 │
│ trial only      │   414 │  470.33 │   1.136 │
└─────────────────┴───────┴─────────┴─────────┘



In [37]:
con.sql("""
SELECT CASE WHEN input_tokens < 2000 THEN 'a <2k'
            WHEN input_tokens < 5000 THEN 'b 2-5k'
            WHEN input_tokens < 10000 THEN 'c 5-10k'
            ELSE 'd 10k+' END AS bucket,
       COUNT(*) n, ROUND(SUM(usd),2) AS usd
FROM cost
WHERE feature='summarize' AND d BETWEEN DATE '2026-07-01' AND DATE '2026-07-31'
GROUP BY 1 ORDER BY 1
""").show()

┌─────────┬───────┬─────────┐
│ bucket  │   n   │   usd   │
│ varchar │ int64 │ double  │
├─────────┼───────┼─────────┤
│ b 2-5k  │  5201 │  121.36 │
│ c 5-10k │ 21214 │  747.55 │
│ d 10k+  │ 15792 │ 1281.54 │
└─────────┴───────┴─────────┘



83% of cost comes from users who never subscribed. Summaries under 10k tokens cost about \\$ 869 a month. If nimbus-mini could handle them, the saving is up to about \\$ 825 (an estimate, and it needs a quality test first). Timed-out calls cost $97.89 a month, which is small but easy to go after.

## 4. Experiment: onboarding_v2
### 4a. Setup checks
Eligible means iOS or Android, account created Jun 15 to Jul 15 UTC.

In [39]:
con.sql("DESCRIBE experiment_assignments").show()
con.sql("SELECT * FROM experiment_assignments LIMIT 8").show(max_width=200)

┌─────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │       column_type        │  null   │   key   │ default │  extra  │
│   varchar   │         varchar          │ varchar │ varchar │ varchar │ varchar │
├─────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ user_id     │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ experiment  │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ variant     │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ assigned_ts │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴──────────────────────────┴─────────┴─────────┴─────────┴─────────┘

┌────────────────┬───────────────┬───────────┬────────────────────────────┐
│    user_id     │  experiment   │  variant  │        assigned_ts         │
│    varchar     │    varchar    │  varchar  │  timestamp with time zone  │
├────────────────┼──────

In [41]:
con.sql("SET TimeZone='UTC'")

# 1. Assignment sanity: rows vs distinct users
con.sql("""
SELECT experiment, variant, COUNT(*) AS n_rows, COUNT(DISTINCT user_id) AS n_users
FROM experiment_assignments GROUP BY 1,2 ORDER BY 1,2
""").show()

con.sql("""
SELECT COUNT(*) AS users_with_multiple_rows FROM (
  SELECT user_id FROM experiment_assignments
  WHERE experiment='onboarding_v2' GROUP BY 1 HAVING COUNT(*)>1)
""").show()

# 2. Base view: join to users, flag eligibility
con.sql("""
CREATE OR REPLACE TEMP VIEW expu AS
SELECT a.user_id, a.variant, a.assigned_ts, u.account_created_ts AS t0,
       u.platform, u.country, (u.anon_id IS NOT NULL) AS had_anon,
       (u.user_id IS NOT NULL) AS in_users,
       (u.platform IN ('ios','android')
        AND u.account_created_ts >= TIMESTAMPTZ '2026-06-15 00:00:00+00'
        AND u.account_created_ts <  TIMESTAMPTZ '2026-07-16 00:00:00+00') AS eligible
FROM experiment_assignments a LEFT JOIN users u USING (user_id)
WHERE a.experiment='onboarding_v2'
""")

con.sql("""
SELECT eligible, in_users, variant, COUNT(*) AS n
FROM expu GROUP BY 1,2,3 ORDER BY 1,2,3
""").show()

# 3. Balance between arms (eligible users only)
con.sql("""
SELECT 'platform' AS dim, platform AS val, variant, COUNT(*) AS n FROM expu WHERE eligible GROUP BY 1,2,3
UNION ALL
SELECT 'had_anon', CAST(had_anon AS VARCHAR), variant, COUNT(*) FROM expu WHERE eligible GROUP BY 1,2,3
ORDER BY 1,2,3
""").show()

# 4. Gap between account creation and assignment
con.sql("""
SELECT variant,
       ROUND(MEDIAN(EPOCH(assigned_ts - t0)),0) AS median_gap_sec,
       ROUND(MAX(ABS(EPOCH(assigned_ts - t0))),0) AS max_abs_gap_sec
FROM expu WHERE eligible GROUP BY 1
""").show()

┌───────────────┬───────────┬────────┬─────────┐
│  experiment   │  variant  │ n_rows │ n_users │
│    varchar    │  varchar  │ int64  │  int64  │
├───────────────┼───────────┼────────┼─────────┤
│ onboarding_v2 │ control   │   2404 │    2404 │
│ onboarding_v2 │ treatment │   2160 │    2160 │
└───────────────┴───────────┴────────┴─────────┘

┌──────────────────────────┐
│ users_with_multiple_rows │
│          int64           │
├──────────────────────────┤
│                        0 │
└──────────────────────────┘

┌──────────┬──────────┬───────────┬───────┐
│ eligible │ in_users │  variant  │   n   │
│ boolean  │ boolean  │  varchar  │ int64 │
├──────────┼──────────┼───────────┼───────┤
│ true     │ true     │ control   │  2404 │
│ true     │ true     │ treatment │  2160 │
└──────────┴──────────┴───────────┴───────┘

┌──────────┬─────────┬───────────┬───────┐
│   dim    │   val   │  variant  │   n   │
│ varchar  │ varchar │  varchar  │ int64 │
├──────────┼─────────┼───────────┼───────┤


Control 2,404 vs treatment 2,160 fails the sample ratio check (p = 0.0003). iOS and Android 4.2.1 are fully assigned, so the problem is Android 4.1.3 (see 4d). Arms look balanced on had_anon, and assignment happens within 20 seconds of signup.

### 4b. Reproducing the PM's numbers
Day n is the n-th 24-hour block after account creation. Clean means a successful, non-demo event.

In [43]:
con.sql("""
CREATE OR REPLACE TEMP VIEW act AS
SELECT x.user_id, x.variant, x.platform, x.had_anon,
  CAST(FLOOR(EPOCH(e.event_ts - x.t0)/86400) AS INT) AS day_n,
  (e.status='success' AND e.feature<>'onboarding_demo') AS clean
FROM expu x JOIN usage_events e ON e.user_id = x.user_id
WHERE x.eligible AND e.event_ts >= x.t0
""")

con.sql("""
SELECT x.variant, COUNT(*) AS users,
  ROUND(100.0*AVG(CASE WHEN a.d1_any   THEN 1 ELSE 0 END),1) AS d1_pm_style_any_event,
  ROUND(100.0*AVG(CASE WHEN a.d1_clean THEN 1 ELSE 0 END),1) AS d1_clean,
  ROUND(100.0*AVG(CASE WHEN a.d713_any   THEN 1 ELSE 0 END),1) AS d7_13_any_event,
  ROUND(100.0*AVG(CASE WHEN a.d713_clean THEN 1 ELSE 0 END),1) AS d7_13_clean
FROM expu x LEFT JOIN (
  SELECT user_id,
    BOOL_OR(day_n=1) AS d1_any,
    BOOL_OR(day_n=1 AND clean) AS d1_clean,
    BOOL_OR(day_n BETWEEN 7 AND 13) AS d713_any,
    BOOL_OR(day_n BETWEEN 7 AND 13 AND clean) AS d713_clean
  FROM act GROUP BY 1) a USING (user_id)
WHERE x.eligible GROUP BY 1 ORDER BY 1
""").show(max_width=200)

┌───────────┬───────┬───────────────────────┬──────────┬─────────────────┬─────────────┐
│  variant  │ users │ d1_pm_style_any_event │ d1_clean │ d7_13_any_event │ d7_13_clean │
│  varchar  │ int64 │        double         │  double  │     double      │   double    │
├───────────┼───────┼───────────────────────┼──────────┼─────────────────┼─────────────┤
│ control   │  2404 │                  41.9 │     41.8 │            34.0 │        34.0 │
│ treatment │  2160 │                  51.0 │     51.0 │            35.7 │        35.7 │
└───────────┴───────┴───────────────────────┴──────────┴─────────────────┴─────────────┘



I reproduced the PM's D1 exactly, 41.9% vs 51.0%. I expected the demo events to inflate it, and they don't (41.8% vs 51.0% without them). So the D1 gain is real. The problem is that the pre-registered metric is D7-13.

### 4c. The pre-registered metric, with confidence intervals
Sample ratio checks and two-proportion tests, overall and by platform and app version.

In [46]:
con.sql("SET TimeZone='UTC'")
ret = con.sql("""
SELECT x.user_id, x.variant, x.platform, x.t0, u.app_version,
       COALESCE(a.d1,false) AS d1, COALESCE(a.d713,false) AS d713
FROM expu x JOIN users u USING (user_id)
LEFT JOIN (SELECT user_id,
             BOOL_OR(day_n=1 AND clean) AS d1,
             BOOL_OR(day_n BETWEEN 7 AND 13 AND clean) AS d713
           FROM act GROUP BY 1) a USING (user_id)
WHERE x.eligible
""").df()

import pandas as pd
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

def srm(df):
    c = df.variant.value_counts()
    return c.to_dict(), chisquare([c['control'], c['treatment']]).pvalue

def lift(df, col):
    t = df[df.variant=='treatment'][col]; c = df[df.variant=='control'][col]
    z, p = proportions_ztest([t.sum(), c.sum()], [len(t), len(c)])
    lo, hi = confint_proportions_2indep(t.sum(), len(t), c.sum(), len(c), compare='diff')
    return dict(n_t=len(t), n_c=len(c), ctrl=round(c.mean(),4), trt=round(t.mean(),4),
                diff=round(t.mean()-c.mean(),4), ci=(round(lo,4), round(hi,4)), p=round(p,4))

print("SRM all:", srm(ret))
for plat in ['ios','android']:
    d = ret[ret.platform==plat]
    print(plat, "SRM:", srm(d))
    print(plat, "D1:", lift(d,'d1'))
    print(plat, "D7-13:", lift(d,'d713'))
print("ALL D1:", lift(ret,'d1'))
print("ALL D7-13:", lift(ret,'d713'))

# Where did the Android treatment users go?
print(pd.crosstab([ret[ret.platform=='android'].app_version],
                  ret[ret.platform=='android'].variant))
ret['wk'] = pd.to_datetime(ret.t0).dt.tz_localize(None).dt.to_period('W').astype(str)
print(pd.crosstab([ret[ret.platform=='android'].wk],
                  ret[ret.platform=='android'].variant))

SRM all: ({'control': 2404, 'treatment': 2160}, 0.0003041442581704827)
ios SRM: ({'treatment': 1485, 'control': 1442}, 0.42673066453656694)
ios D1: {'n_t': 1485, 'n_c': 1442, 'ctrl': 0.4542, 'trt': 0.5508, 'diff': 0.0966, 'ci': (0.0604, 0.1324), 'p': 0.0}
ios D7-13: {'n_t': 1485, 'n_c': 1442, 'ctrl': 0.3426, 'trt': 0.3865, 'diff': 0.044, 'ci': (0.0091, 0.0787), 'p': 0.0135}
android SRM: ({'control': 962, 'treatment': 675}, 1.3080853214691669e-12)
android D1: {'n_t': 675, 'n_c': 962, 'ctrl': 0.3649, 'trt': 0.4193, 'diff': 0.0544, 'ci': (0.0064, 0.1024), 'p': 0.0262}
android D7-13: {'n_t': 675, 'n_c': 962, 'ctrl': 0.3358, 'trt': 0.2933, 'diff': -0.0424, 'ci': (-0.0874, 0.0035), 'p': 0.0697}
ALL D1: {'n_t': 2160, 'n_c': 2404, 'ctrl': 0.4185, 'trt': 0.5097, 'diff': 0.0913, 'ci': (0.0623, 0.12), 'p': 0.0}
ALL D7-13: {'n_t': 2160, 'n_c': 2404, 'ctrl': 0.3399, 'trt': 0.3574, 'diff': 0.0176, 'ci': (-0.0101, 0.0452), 'p': 0.2138}
variant      control  treatment
app_version                    
4

D7-13 overall is +1.8 points (CI -1.0 to +4.5, p = 0.21), not significant. iOS is +4.4 (CI +0.9 to +7.9), but I only looked at iOS after seeing the data. Android is -4.2, and its split is broken.

### 4d. Missing users and the cost of the demo

In [48]:
# 1. Are eligible users missing from the assignment table? (find the lost treatment users)
con.sql("""
SELECT u.platform, u.app_version,
       COUNT(*) AS users_in_window,
       COUNT(a.user_id) AS assigned,
       COUNT(*) - COUNT(a.user_id) AS never_assigned
FROM users u LEFT JOIN experiment_assignments a
  ON a.user_id=u.user_id AND a.experiment='onboarding_v2'
WHERE u.platform IN ('ios','android')
  AND u.account_created_ts >= TIMESTAMPTZ '2026-06-15 00:00:00+00'
  AND u.account_created_ts <  TIMESTAMPTZ '2026-07-16 00:00:00+00'
GROUP BY 1,2 ORDER BY 1,2
""").show()

# 2. Android D7-13 by app version (arms are still unbalanced, read with care)
print(ret[ret.platform=='android'].groupby(['app_version','variant']).d713.agg(['mean','count']))

# 3. Demo cost guardrail: cost of onboarding_demo events per treated user
con.sql("""
SELECT x.variant, COUNT(DISTINCT x.user_id) AS users,
       ROUND(SUM(c.usd),2) AS demo_usd,
       ROUND(SUM(c.usd)/COUNT(DISTINCT x.user_id),4) AS demo_usd_per_user
FROM expu x LEFT JOIN cost c
  ON c.user_id=x.user_id AND c.feature='onboarding_demo'
WHERE x.eligible GROUP BY 1
""").show()

┌──────────┬─────────────┬─────────────────┬──────────┬────────────────┐
│ platform │ app_version │ users_in_window │ assigned │ never_assigned │
│ varchar  │   varchar   │      int64      │  int64   │     int64      │
├──────────┼─────────────┼─────────────────┼──────────┼────────────────┤
│ android  │ 4.1.3       │             738 │      529 │            209 │
│ android  │ 4.2.1       │            1108 │     1108 │              0 │
│ ios      │ 5.0.2       │            2927 │     2927 │              0 │
└──────────┴─────────────┴─────────────────┴──────────┴────────────────┘

                           mean  count
app_version variant                   
4.1.3       control    0.297368    380
            treatment  0.241611    149
4.2.1       control    0.360825    582
            treatment  0.307985    526
┌───────────┬───────┬──────────┬───────────────────┐
│  variant  │ users │ demo_usd │ demo_usd_per_user │
│  varchar  │ int64 │  double  │      double       │
├───────────┼───────┼─

209 eligible Android 4.1.3 users were never assigned. That is the sample ratio problem, and I can't tell which arm they would have landed in. The demo costs \\$ 0.016 per treated user ($34.31 in total).

### 4e. Android on the cleaner app version

In [51]:
a = ret[(ret.platform=='android') & (ret.app_version=='4.2.1')]
print(lift(a, 'd713'))

{'n_t': 526, 'n_c': 582, 'ctrl': 0.3608, 'trt': 0.308, 'diff': -0.0528, 'ci': (-0.1078, 0.0029), 'p': 0.0629}


**Takeaway:** Android 4.2.1 alone is -5.3 points (CI -10.8 to +0.3, p = 0.063). Not significant, but it points the same way as the rest of Android, so the dip isn't only the 4.1.3 bug.

I think we shouldn't ship 100%. Try to fix the 4.1.3 assignment bug and re-run on D7-13 with a clean 50/50 split. An iOS-only rollout with a 10% holdback is a low-risk option. Hold Android until the dip is explained.

## 5. Limits and next steps
- The dashboard replica is inferred from outputs. I never saw the legacy query.
- The iOS result is a post-hoc subgroup, one of three looks at the data.
- I can't explain the June gap of about $87. My guess is the same timeouts, duplicates and rounding that apply every month.
- I didn't analyze returning users (anon_id) separately. I only checked that both arms had a similar share.
- Tools: I used an AI assistant for restructing my code better and for rephrasing some of these notes to make it more professional. I ran every query and checked the numbers against the outputs myself.

## Appendix A: lessons and pitfalls
All amounts are in USD. Summary: 8 wrong assumptions, 5 mistakes unrelated to the analysis, and one change in approach.

### A1. Where my assumptions were wrong

**Part 1: Cost reconciliation**

| # | I assumed | What the data showed | Fix |
|---|---|---|---|
| 1 | The dashboard counts every status | My replica gave 6,461.81 against the real 6,310.49 (151 too high). The dashboard counts successes only. | Compared by feature under three status filters |
| 2 | Bare dates are fine for month filters | Timestamps had a local offset, so month edges were 6 hours off | UTC session and explicit TIMESTAMPTZ bounds |
| 3 | Early guesses: unpriced model about 420, duplicates about -38, timeouts about 93 | Actual: +417.80, -110.81, +97.89 (duplicates were nearly 3x my guess) | Used only query output |

**Part 2: Unit economics**

| # | I assumed | What the data showed | Fix |
|---|---|---|---|
| 4 | Capping heavy users would save money | The top 1% of users are only 7.4% of cost | Switched to the nimbus-mini and free-tier ideas |
| 5 | Moving 30% of summaries to mini saves about 610 | Cost sits in the larger inputs, so it depends on which calls move | Sized by input-size bucket: up to about 825 |

**Part 3: Experiment**

| # | I assumed | What the data showed | Fix |
|---|---|---|---|
| 6 | Demo events inflate D1 | Excluding them moves D1 by 0.1 point | Dropped the claim. The PM's issue is the metric. |
| 7 | About 290 Android treatment users were missing | Shortfall is about 143. 209 eligible 4.1.3 users were never assigned. | Traced it to app version 4.1.3 |
| 8 | Android is unreadable because of 4.1.3 | 4.2.1 is also negative (-5.3 points, p = 0.063) | Reported as trending negative, not proven |

### A2. Mistakes unrelated to the analysis

| Type | What happened | Fix |
|---|---|---|
| Code | `rows` is reserved in DuckDB, so it can't be an alias | Renamed to `n_rows` |
| Code | First cost view lacked request_id and input_tokens, so a join failed | Rebuilt the view with both |
| Code | Guessed column names before DESCRIBE and summed June with July | Checked columns, filtered to July |
| Setup | DuckDB created an empty 12 KB file because the real one wasn't downloaded | Downloaded the data and reconnected |
| Setup | An open connection blocked `read_only=True` | Restarted the kernel |

### A3. How the approach changed

| | First pass (Appendix B) | Second pass (sections 2b to 2d) |
|---|---|---|
| Start | Guessed how the dashboard works | Started from the invoice |
| Method | Built a replica and walked it to the invoice | Rebuilt each invoice line from the logs until all six unit totals matched |
| Dashboard rules | Assumed | Worked out by category with status filters |
| Result | A 151 gap I couldn't explain | A bridge with no unexplained bar |

### A4. Pitfalls to keep in mind

| Area | Pitfall |
|---|---|
| Bridge | Step sizes depend on order |
| Bridge | Step G (UTC month) came out positive. I measured it and can't explain it. |
| Bridge | Chat and summarization differ from the dashboard by cents. Total residual is 0.05. |
| Metrics | Retention day n is a 24-hour block after signup, not a calendar day |
| Metrics | First-week cost only counts usage tied to a user_id |
| Revenue | Revenue is cash. The 39.5% and 71% figures are the two ends of the range. |
| Estimates | The 825 saving assumes mini saves about 95% per call and quality holds |ssumes mini saves about 95% per call and quality holds |ssumes mini saves about 95% per call and quality holds |

## Appendix B: first bridge attempt
This replica counted every status and overshot the dashboard by $151. Section 2c found the cause. I'm keeping it so the change in approach is visible.

In [56]:
con.sql("SET TimeZone='UTC'")
con.sql("""
WITH b AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ingested_ts) AS rn,
    CAST(timezone('America/Los_Angeles', event_ts) AS DATE) AS pt_date,
    CASE WHEN model LIKE 'nimbus-large%' THEN 'nimbus-large' ELSE model END AS bm
  FROM usage_events
),
c AS (
  SELECT *,
    COALESCE(CASE WHEN provider='vox' THEN audio_seconds/60.0*0.006 END, 0) AS vox_exact,
    COALESCE(CASE WHEN provider='vox' THEN CEIL(audio_seconds/15.0)*15/60.0*0.006 END, 0) AS vox_round,
    COALESCE(CASE bm WHEN 'nimbus-large' THEN 3.0 WHEN 'nimbus-mini' THEN 0.15
                     WHEN 'nimbus-embed' THEN 0.02 END / 1e6 * input_tokens, 0) AS in_cost,
    COALESCE(CASE bm WHEN 'nimbus-large' THEN 15.0 WHEN 'nimbus-mini' THEN 0.6
                     ELSE 0 END / 1e6 * output_tokens, 0) AS out_cost,
    CASE WHEN bm='nimbus-large' AND event_ts >= TIMESTAMPTZ '2026-07-15 00:00:00+00'
         THEN 3.0/1e6*COALESCE(output_tokens,0) ELSE 0 END AS out_uplift,
    (pt_date BETWEEN DATE '2026-07-01' AND DATE '2026-07-31') AS pt_july,
    (event_ts >= TIMESTAMPTZ '2026-07-01 00:00:00+00'
     AND event_ts < TIMESTAMPTZ '2026-08-01 00:00:00+00') AS utc_july
  FROM b
)
SELECT
 ROUND(SUM(vox_exact+in_cost+out_cost) FILTER (WHERE pt_july AND model<>'nimbus-large-0526'),2) AS A_replica_guess,
 ROUND(SUM(vox_exact+in_cost+out_cost) FILTER (WHERE pt_july),2)                                 AS B_add_0526,
 ROUND(SUM(vox_exact+in_cost+out_cost+out_uplift) FILTER (WHERE pt_july),2)                      AS C_add_price_change,
 ROUND(SUM(vox_exact+in_cost+out_cost+out_uplift) FILTER (WHERE pt_july AND rn=1),2)             AS D_dedupe,
 ROUND(SUM(vox_exact+in_cost+out_cost+out_uplift) FILTER (WHERE pt_july AND rn=1 AND status IN ('success','timeout')),2) AS E_drop_errors,
 ROUND(SUM(vox_round+in_cost+out_cost+out_uplift) FILTER (WHERE pt_july AND rn=1 AND status IN ('success','timeout')),2) AS F_vox_rounding,
 ROUND(SUM(vox_round+in_cost+out_cost+out_uplift) FILTER (WHERE utc_july AND rn=1 AND status IN ('success','timeout')),2) AS G_utc_month
FROM c
""").show(max_width=250)

┌─────────────────┬────────────┬────────────────────┬──────────┬───────────────┬────────────────┬─────────────┐
│ A_replica_guess │ B_add_0526 │ C_add_price_change │ D_dedupe │ E_drop_errors │ F_vox_rounding │ G_utc_month │
│     double      │   double   │       double       │  double  │    double     │     double     │   double    │
├─────────────────┼────────────┼────────────────────┼──────────┼───────────────┼────────────────┼─────────────┤
│         6461.81 │    6889.77 │            6951.82 │  6837.25 │       6777.47 │        6808.88 │     6860.56 │
└─────────────────┴────────────┴────────────────────┴──────────┴───────────────┴────────────────┴─────────────┘



## 6. Part 4: Making this a daily pipeline

### What it would do every day
- Rebuild AI cost from the raw logs using the rules the invoices follow: one row per request_id, count successes and timeouts, round Vox up to 15 seconds per request, use UTC days, and look prices up by effective date.
- Compare that number with the dashboard by provider and category. If they differ by more than 2% on any day, send an alert.
- Keep the pricing table versioned, with start and end dates and a note on where each price came from. No quiet manual edits.

### What I'd alert on
- Any model in the logs that has no price. This would have caught nimbus-large-0526 on July 10. The pipeline should fail loudly or show an "unpriced" line, never drop the rows.
- Jumps in the duplicate rate, timeout share or error share, compared with the last 7 days.
- Ingestion lag over 15 minutes, or a spike in missing token counts or audio seconds, split by app version.
- Daily cost more than 3 standard deviations from the last 4 weeks, and cost per active user outside its normal range.
- At month end, our total against the invoice. Anything off by more than 1% gets looked at that day.

### Spotting a price change before the invoice
Both providers show daily usage and spend. Every day, divide their reported spend by our logged units, per model and separately for input and output tokens. That gives the rate we're actually paying. If it drifts more than 3% from our pricing table, alert. The July 15 nimbus-large output change (15 to 18 per million tokens) would have shown up as a step that day, not three weeks later on the invoice. I'd also have someone skim the providers' pricing pages and changelogs. It's cheap, and sometimes they warn you first.

### For experiments
- Run a sample ratio check daily, overall and by platform and app version. It would have flagged Android 4.1.3 within days.
- Check assignment coverage: how many eligible users never got assigned.
- Write down the main metric and the analysis window before launch, and show cost per treated user next to it.

### Ownership
Give the pipeline one named owner, keep the queries in version control, and make the tests fail the daily run instead of logging a warning nobody reads. Once it matches the old dashboard, retire the old query. Nobody on the team wrote it or understands it.